In [11]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough



# Load pdf file


### PyPDFLoader was chosen because:

1. Simple API - just pass file path and call .load()
2. Extracts selectable text directly from PDFs
3. Preserves metadata (page numbers, source file info)
4. Well-integrated with LangChain ecosystem
5. Good for text-based PDFs (not scanned images)

##### Alternative PDF loaders in LangChain:

- PDFPlumberLoader: Better text extraction, handles tables/layouts
- PDFMinerLoader: More robust parsing, slower
- UnstructuredPDFLoader: Handles complex layouts, images, tables
- AmazonTextractPDFLoader: AWS-based, handles scanned PDFs with OCR
- LlamaParseLoader: AI-powered extraction, excellent for complex PDFs

##### Trade-offs:

PyPDFLoader: Fast, simple, lightweight - best for clean text PDFs
Others: More features but slower, more dependencies, higher costs (some)

For this project, PyPDFLoader is ideal because:

- Academic paper is text-based and clean
- Performance is good for RAG workflows
- Minimal setup needed for proof-of-concept

print("PDF Loader comparison documented. Using PyPDFLoader for this academic paper.")


In [2]:
#loading data
loader = PyPDFLoader("data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf")
document = loader.load()
print(document)

Ignoring wrong pointing object 18 0 (offset 0)


[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creator': 'Preview', 'creationdate': "D:20240909152042Z00'00'", 'author': 'Thu Vu', 'moddate': "D:20240910141854Z00'00'", 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology', 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='APPLIED COGNITIVE PSYCHOLOGY\nAppl. Cognit. Psychol. 20: 139–156 (2006)\nPublished online 31 October 2005 in Wiley InterScience\n(www.interscience.wiley.com) DOI: 10.1002/acp.1178\nConsequences of Erudite Vernacular Utilized Irrespective\nof Necessity: Problems with Using Long Words Needlessly\nDANIEL M. OPPENHEIMER*\nPrinceton University, USA\nSUMMARY\nMost texts on writing style encourage authors to avoid overly-complex words. However, a majority\nof undergraduates admit to deliberately increasing the complexity of their vocabulary so as to give\nthe impression of intelligen

# Split text


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, 
                                               chunk_overlap=120,
                                               length_function=len,
                                            separators=["\n\n", "\n", " "])
doc = text_splitter.split_documents(document)
print(len(doc))
print(doc)

13
[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creator': 'Preview', 'creationdate': "D:20240909152042Z00'00'", 'author': 'Thu Vu', 'moddate': "D:20240910141854Z00'00'", 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology', 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='APPLIED COGNITIVE PSYCHOLOGY\nAppl. Cognit. Psychol. 20: 139–156 (2006)\nPublished online 31 October 2005 in Wiley InterScience\n(www.interscience.wiley.com) DOI: 10.1002/acp.1178\nConsequences of Erudite Vernacular Utilized Irrespective\nof Necessity: Problems with Using Long Words Needlessly\nDANIEL M. OPPENHEIMER*\nPrinceton University, USA\nSUMMARY\nMost texts on writing style encourage authors to avoid overly-complex words. However, a majority\nof undergraduates admit to deliberately increasing the complexity of their vocabulary so as to give\nthe impression of intelli

### Create embeddings


#### Understanding Embeddings: A Beginner's Guide

##### What are Embeddings?

Embeddings are numerical representations of text converted into vectors (lists of numbers). Think of them as a way to translate words, sentences, or documents into a language that machine learning models can understand and process.

**Example:**

Notice how "cat" and "dog" embeddings are very similar because they represent similar concepts.

---

#### Why Do We Need Embeddings?

1. **Semantic Understanding**: Embeddings capture the meaning of words. Similar words have similar embeddings.
2. **Machine Learning Ready**: Neural networks work with numbers, not raw text. Embeddings convert text to numbers.
3. **Similarity Search**: Find related documents quickly using vector distance (e.g., which documents are similar to a query?).
4. **Memory Efficient**: Instead of storing entire documents, store compressed numerical representations.
5. **RAG (Retrieval-Augmented Generation)**: Essential for retrieving relevant context from large documents to feed into LLMs.

---

#### How Are Embeddings Created?

##### The Process:

1. **Input Text**: You provide text (word, sentence, or document)
2. **Neural Network**: A pre-trained deep learning model processes the text
3. **Output Vector**: The model outputs a vector of numbers (e.g., 384 dimensions)
4. **Storage**: Store these vectors in a vector database for fast retrieval


In [18]:
embeddings =HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 788.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Store in vector DB


In [19]:
import uuid

def create_vectorstore(chunks, embedding_function, vectorstore_path):

    # Create a list of unique ids for each document based on the content
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    
    # Ensure that only unique docs with unique ids are kept
    unique_ids = set()
    unique_chunks = []
    
    unique_chunks = [] 
    for chunk, id in zip(chunks, ids):     
        if id not in unique_ids:       
            unique_ids.add(id)
            unique_chunks.append(chunk) 

    # Create a new Chroma database from the documents
    vectorstore = Chroma.from_documents(documents=unique_chunks, 
                                        ids=list(unique_ids),
                                        embedding=embeddings, 
                                        persist_directory = vectorstore_path)

    vectorstore.persist()
    
    return vectorstore



In [20]:

# Create vectorstore
vectorstore = create_vectorstore(chunks= doc, 
                                 embedding_function=embeddings,
                                 vectorstore_path="vectorstore_test")

In [21]:
# Create retriever and get relevant chunks
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20, "lambda_mult": 0.5},
)
relevant_chunks = retriever.invoke("What is the title of the paper?")
relevant_chunks



[Document(metadata={'total_pages': 3, 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'moddate': "D:20240910141854Z00'00'", 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology', 'page_label': '1', 'page': 0, 'author': 'Thu Vu', 'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creationdate': "D:20240909152042Z00'00'", 'creator': 'Preview'}, page_content='When it comes to writing, most experts agree that clarity, simplicity and parsimony are\nideals that authors should strive for. In their classic manual of style, Strunk and White\n(1979) encourage authors to ‘omit needless words.’ Daryl Bem’s (1995) guidelines for\nsubmission to Psychological Bulletin advise, ‘the ﬁrst step towards clarity is writing\nsimply.’ Even the APA publication manual (1996) recommends, ‘direct, declarative\nsentences with simple common words are usually best.’\nHowever, most of us can likely recall having read papers, either by colleagues or\nstudents,

In [8]:
# Prompt template
PROMPT_TEMPLATE = """
You are an assistant for question-answering tasks.
Use only the retrieved context to answer the question.
If the answer is not in the context, reply exactly: I don't know based on this document.
Keep the answer concise and include page citations like [p.X].

Context:
{context}

Question: {question}
Answer:
"""



In [22]:
# Concatenate context text
context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, 
                                question="What is the title of the paper?")
print(prompt)

Human: 
    You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer
    the question. If you don't know the answer, say that you
    don't know. DON'T MAKE UP ANYTHING.
When it comes to writing, most experts agree that clarity, simplicity and parsimony are
ideals that authors should strive for. In their classic manual of style, Strunk and White
(1979) encourage authors to ‘omit needless words.’ Daryl Bem’s (1995) guidelines for
submission to Psychological Bulletin advise, ‘the ﬁrst step towards clarity is writing
simply.’ Even the APA publication manual (1996) recommends, ‘direct, declarative
sentences with simple common words are usually best.’
However, most of us can likely recall having read papers, either by colleagues or
students, in which the author appears to be deliberately using overly complex words.
Experience suggests that the experts’ advice contrasts with prevailing wisdom on how to
sound more intelligent as a writer. 

### use local LLM


In [23]:
llm = OllamaLLM(model="gemma3:4b")

In [24]:
llm.invoke(prompt)

"The context doesn't provide the title of the paper. It describes the study design and results."

In [15]:
#Create RAG Chain
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [16]:
# ask a question
answer = qa.invoke("what is the title of the paper?")
print(answer)

{'query': 'what is the title of the paper?', 'result': 'The strategy of complex-writing'}


In [17]:
# Using LangChain Expression Language + title router

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def extract_title_from_first_page(pages):
    if not pages:
        return None

    first_page_doc = next((d for d in pages if d.metadata.get("page", 0) == 0), pages[0])
    lines = [line.strip() for line in first_page_doc.page_content.splitlines() if line.strip()]

    # Remove obvious section headers/noise and keep likely title lines.
    stop_headers = {"abstract", "keywords", "introduction"}
    title_lines = []
    for line in lines[:30]:
        clean = re.sub(r"\s+", " ", line).strip()
        low = clean.lower().strip(":")

        if low in stop_headers:
            break
        if len(clean) < 12 or len(clean) > 220:
            continue
        if clean.isdigit():
            continue

        title_lines.append(clean)
        if len(" ".join(title_lines)) >= 45:
            break

    if not title_lines:
        return None

    title = re.sub(r"\s+", " ", " ".join(title_lines)).strip(" -")
    return title if title else None


# For normal questions, keep LCEL flow.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)


def ask_pdf(question):
    q = question.lower()
    title_question_signals = [
        "title",
        "name of the paper",
        "paper name",
        "name of this paper",
        "article title",
    ]

    if any(signal in q for signal in title_question_signals):
        title = extract_title_from_first_page(document)
        if title:
            return {
                "answer": f"{title} [p.1]",
                "route": "title_extractor",
                "source_pages": [1],
            }

    docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    source_pages = sorted(
        {
            (d.metadata.get("page") + 1) if isinstance(d.metadata.get("page"), int) else d.metadata.get("page")
            for d in docs
        }
    )

    return {
        "answer": answer,
        "route": "retriever",
        "source_pages": source_pages,
    }


result = ask_pdf("What's the title of this paper?")
print(result["answer"])
print(result)


'The present paper provides an empirical investigation of the strategy of complexity, and finds such a strategy to be unsuccessful.'